In [5]:
import pandas as pd

In [6]:
GEX_SETS_PATH = "/home/mariya/Development/pallas/reports/output/cp/rf_GEX_20260606_190416/FNRCoP/2026-06-24_19-50-20/prediction_sets.csv"
DNAM_SETS_PATH = "/home/mariya/Development/pallas/reports/output/cp/rf_DNAm_20260606_193214/FNRCoP/2026-06-24_19-51-55/prediction_sets.csv"

In [7]:
# Read them in as df
gex_sets = pd.read_csv(GEX_SETS_PATH, index_col=0)
dnam_sets = pd.read_csv(DNAM_SETS_PATH, index_col=0)
gex_sets.head()

,id,predicted_label,prediction_sets,true_label,prediction_set_softmax_scores,known_class_softmax_scores
dataset,,,,,,
test,SJHBBBM5PY5,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.5266792135775378,0.526679
test,SJHBBBM5TXY,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.6913082574568313,0.691308
test,SJHBBBM67H6,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.4145710304682025,0.414571
test,ALLT-330,"AML, KMT2A-r","AML, KMT2A-r;B-ALL, KMT2A-r","B-ALL, ETV6::RUNX1",0.2012814793797356;0.1696828955378148,0.021721
test,ALLT-339,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, ETV6::RUNX1",0.2792603597484029,0.130439


In [8]:
import pandas as pd
import plotly.graph_objects as go
from collections import defaultdict

# # ── Example data – replace with your own df ──────────────────────────────────
# df = pd.DataFrame({
#     "true_label":      ["cat", "cat", "dog", "dog", "bird", "bird", "cat"],
#     "prediction_sets": ["cat,dog", "cat", "dog,cat", "dog", "bird,cat", "bird,dog", "dog,bird"],
# })
# # ─────────────────────────────────────────────────────────────────────────────

def build_sankey(df: pd.DataFrame,
                 true_col: str = "true_label",
                 pred_col: str = "prediction_sets") -> go.Figure:
    """
    Build a Sankey diagram mapping each true label to all members of its
    prediction_sets (comma-separated string).

    Left nodes  = true labels
    Right nodes = predicted labels (suffixed with ' (pred)' to keep them
                  visually distinct even when the label text is the same)
    """
    # Count flows: (true_label, predicted_label) → count
    flow_counts: dict[tuple[str, str], int] = defaultdict(int)

    for _, row in df.iterrows():
        true = str(row[true_col]).strip()
        preds = [p.strip() for p in str(row[pred_col]).split(";") if p.strip()]
        for pred in preds:
            flow_counts[(true, pred)] += 1

    # Build node lists (left nodes first, then right nodes)
    true_labels = sorted({k[0] for k in flow_counts})
    pred_labels = sorted({k[1] for k in flow_counts})

    # Right-side node labels get a suffix so identical names don't collapse
    pred_labels_display = [f"{p} (pred)" for p in pred_labels]

    all_labels = true_labels + pred_labels_display
    label_index = {label: i for i, label in enumerate(all_labels)}

    sources, targets, values = [], [], []
    for (true, pred), count in flow_counts.items():
        sources.append(label_index[true])
        targets.append(label_index[f"{pred} (pred)"])
        values.append(count)

    # Colour palette (cycles if more labels than colours)
    palette = [
        "#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A",
        "#19D3F3", "#FF6692", "#B6E880", "#FF97FF", "#FECB52",
    ]
    node_colors = [palette[i % len(palette)] for i in range(len(all_labels))]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=20,
            thickness=20,
            line=dict(color="white", width=0.5),
            label=all_labels,
            color=node_colors,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
        ),
    ))

    fig.update_layout(
        title_text="True Labels → Prediction Sets",
        title_font_size=18,
        font_size=13,
        height=500 + 40 * max(len(true_labels), len(pred_labels)),
    )
    return fig


fig = build_sankey(gex_sets)
fig.show()                          # opens in browser
# fig.write_html("sankey.html")     # uncomment to save as HTML
# fig.write_image("sankey.png")     # uncomment to save as PNG (needs kaleido)